In [ ]:
!pip install swig
!pip install gymnasium[box2d]
!pip install pyglet==1.5.27

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 23.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp312-cp312-linux_x86_64.whl size=2398999 sha256=d8d9e94e5265a5474e8ac4e1907a60964be4f51b80acdc258406b9f61cbce3f0
  Stored in directory: /root/.cache/pip/wheels/2a/e9/60/774da0bcd07f7dc7761a8590fa2d065e4069568e78dcdc3318
Successfully built box2d-py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Code

## Env

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit
import math
import pyglet
from pyglet import shapes

class CarRacing(gym.Env):

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}

    # __init__()
    def __init__(
        self,
        render_mode: str | None = None,
        continuous: bool = True,
        lap_complete_percent: float = 0.95,
        domain_randomize: bool = False,
        reward_shaping: bool = True,
        max_episode_steps: int = 3000,
        max_laps: int = 1,
    ):
        super(CarRacing, self).__init__()
        self._env = gym.make("CarRacing-v3", render_mode=render_mode, continuous=continuous, lap_complete_percent=lap_complete_percent)
        base_env = self._env.unwrapped
        self._env = TimeLimit(base_env, max_episode_steps=max_episode_steps)

        self.render_mode = render_mode
        self.continuous = continuous
        self.lap_complete_percent = lap_complete_percent
        self._reward_shaping = reward_shaping

        # State S_t
        image_space = spaces.Box(0, 255, shape=(96, 96, 3), dtype=np.uint8)

        # state branch: [d_t, v_t, infield, pitroad, ell_t, w_t, f_t, kappa_t, psi_t, v_lat]
        # psi_t (heading error) and v_lat (lateral speed) help the agent keep orientation and stay centred.
        state_low  = np.array([-5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.05, -math.pi, -20.0], dtype=np.float32)
        state_high = np.array([ 5.0, 70.0, 1.0, 1.0, 1.0, 1.0, 1.0,  0.05,  math.pi,  20.0], dtype=np.float32)
        state_space = spaces.Box(low=state_low, high=state_high, dtype=np.float32)

        self.observation_space = spaces.Dict({"image": image_space, "state": state_space})

        # Action A_t
        if self.continuous:
            self.action_space = spaces.Box(
                low=np.array([-1.0, 0.0, 0.0, 0.0], dtype=np.float32),
                high=np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),
                shape=(4,),
                dtype=np.float32,
            )
        else:
            # 0:nothing, 1:steer right, 2:steer left, 3:gas, 4:brake, 5:pit
            self.action_space = spaces.Discrete(6)

        # Internal states
        self._wear = 0.0   # 0=new, 1=fully_worn
        self._fuel = 1.0   # 1=full, 0=empty
        self._dt = 1.0 / 50.0  # CarRacing runs at 50 frames per second

        # ===== Fuel (per second) ======
        # At idle: drains slowly, but at full gas: drains more (will need to adjust)
        self.fuel_base_per_s = 0.0015         # idle consumption
        self.fuel_full_per_s = 0.0110         # extra at gas=1 (so total ~0.0125/s at speed)

        # Wear (per second)
        self.wear_base_per_s = 0.0003         # always-on tiny wear
        self.wear_brake_per_s = 0.0030        # additional at brake=1
        self.wear_steer_per_s = 0.0020        # additional at steer=1

        # ===== Resource thresholds & shaping =====
        # Termination thresholds
        self.fuel_empty_threshold = 0.02    # below ~2% fuel -> out of fuel
        self.wear_max_threshold = 0.98      # above ~98% wear -> tire failure

        # One-time penalties when we actually fail due to resources
        self.fuel_empty_penalty = 30.0
        self.wear_max_penalty = 20.0

        # Small per-step cost for using resources (only applied if reward_shaping=True)
        self.fuel_cost_per_unit = 2.0       # cost * (fuel_before - fuel_after)
        self.wear_cost_per_unit = 1.0       # cost * (wear_after - wear_before)
        # ========================================

        # Speed scaling reference (m/s); above this speed, scaling is 1.0
        self.speed_ref_mps = 40.0 # Will need to adjust based on testing

        self.progress = 0.0 # overall progress in [0,1] if we want to track
        self._last_progress = 0.0

        self._gauge = None # the gauge window

        # ==== Pitstop configuration ====
        self._pit_center_test = False       # toggle to True to center pit stops and test
        self._pit_speed_max = 5.0           # must be slow to service
        self._pit_count = 25                # number of pit sectors per lap
        self._pit_sector_width = 0.0075     # sector width in progress units

        # Painted strip geometry (world-space)
        self._pit_sign = 1.0
        self._pit_inner_offset = 3.2        # start position relative to centerline
        self._pit_width = 3.5               # thickness of the painted strip

        # Detection must match the paint exactly:
        self._pit_polys_count = 0
        self._prev_pitroad     = False
        self._pit_lock_sector  = None        # prevents spamming within the current sector

        self._sync_pit_bounds()
        # ===============================

        # ==== How much wear impacts steering and gas ====
        self.steering_grip_min = 0.3      # min grip at wear=1.0 (30% of steering)
        self.steering_wear_strength = 0.0 # 1.0 = full effect, < 1.0 = weaker effect

        # Multi-lap state
        self.max_laps = max(1, int(max_laps))
        self._current_lap = 1

    # reset()
    def reset(self, *, seed=None, options=None):
        observation, info = self._env.reset(seed=seed, options=options)

        self.progress = 0.0
        self._last_progress = 0.0

        self._wear = 0.0
        self._fuel = 1.0

        self._prev_pitroad = False

        # reset lap counter
        self._current_lap = 1
        info["lap"] = self._current_lap
        info["max_laps"] = self.max_laps

        # build static painted pit strips onto the track
        try:
            self._sync_pit_bounds()
            self._build_pit_polys()
        except Exception:
            pass

        return self._get_obs(observation), info

    # step()
    def step(self, action):
        # Map our action to base env (3 controls) + pit command
        base_action, pit_command = self._map_action(action)

        # Get actions
        steer, gas, brake = base_action[0], base_action[1], base_action[2]

        # Step in base environment
        observation, reward, terminated, truncated, info = self._env.step(base_action)

        # If the car is off the track surface (and not in the pit),
        # immediately terminate the episode.
        if self._is_infield_from_offset() and not self._is_in_pit():
            terminated = True
            # Optional strong penalty to discourage leaving the track
            reward -= 20.0

        # Get speed from box2d
        velocity = self._get_velocity()
        # print(f"Velocity: {velocity:.2f} | Fuel: {self._fuel:.3f} | Wear: {self._wear:.3f}")

        # Update resources
        steer, gas, brake = float(base_action[0]), float(base_action[1]), float(base_action[2])

        # Scale effects by speed (0.3 at standstill to 1.0 at speed_ref and above)
        speed_scale = 0.3 + 0.7 * min(1.0, velocity / self.speed_ref_mps)

        # Fuel: from idle to full-throttle rate, then scale by speed.
        fuel_rate_per_s = self.fuel_base_per_s + gas * (self.fuel_full_per_s - self.fuel_base_per_s)
        fuel_rate_per_s *= speed_scale
        self._fuel = max(0.0, self._fuel - self._dt * fuel_rate_per_s)

        # Wear: base + brake + steering contributions, then scale by speed.
        wear_rate_per_s = (
            self.wear_base_per_s
            + brake * self.wear_brake_per_s
            + abs(steer) * self.wear_steer_per_s
        )
        wear_rate_per_s *= speed_scale
        self._wear = min(1.0, self._wear + self._dt * wear_rate_per_s)

        # fuel_used = max(0.0, fuel_before - self._fuel)
        # wear_added = max(0.0, self._wear - wear_before)

        # # Optional per-step shaping: small penalty for burning fuel / tires
        # if self._reward_shaping:
        #     reward -= self.fuel_cost_per_unit * fuel_used
        #     reward -= self.wear_cost_per_unit * wear_added

        # ==== Resource-based termination conditions ====
        out_of_fuel = (self._fuel <= self.fuel_empty_threshold)
        tires_gone = (self._wear >= self.wear_max_threshold)

        if out_of_fuel:
            terminated = True
            if self._reward_shaping:
                reward -= self.fuel_empty_penalty

        if tires_gone:
            terminated = True
            if self._reward_shaping:
                reward -= self.wear_max_penalty

        # ==== Multi-lap logic ====
        # per-lap progress on the current track
        ell_t = self._env.unwrapped.tile_visited_count / len(self._env.unwrapped.track)

        env_done = bool(terminated or truncated)
        lap_finished = env_done and (ell_t >= self.lap_complete_percent - 1e-6)

        info["lap"] = self._current_lap
        info["max_laps"] = self.max_laps
        info["lap_finished"] = False
        info["finished_lap"] = None
        info["reset_for_new_lap"] = False

        if lap_finished:
            info["lap_finished"] = True
            info["finished_lap"] = self._current_lap

        if lap_finished and self._current_lap < self.max_laps:
            # Finished a lap but not the whole race: start a new lap
            finished_lap = self._current_lap
            self._current_lap += 1
            info["lap_finished"] = True
            info["finished_lap"] = finished_lap
            info["lap"] = self._current_lap

            # Reset underlying env to start the next lap
            observation, info_reset = self._env.reset()

            # Keep fuel/wear across laps (endurance), but reset some internal race state
            self.progress = 0.0
            self._last_progress = 0.0
            self._prev_pitroad = False
            self._pit_lock_sector = None

            try:
                self._sync_pit_bounds()
                self._build_pit_polys()
            except Exception:
                pass

            # Don't end the episode from the agent's POV
            terminated = False
            truncated = False

            # Recompute progress on the fresh lap
            ell_t = self._env.unwrapped.tile_visited_count / len(self._env.unwrapped.track)
            info["reset_for_new_lap"] = True
        else:
            # Race progress across all laps [0,1]
            if self.max_laps > 0:
                self.progress = ((self._current_lap - 1) + ell_t) / float(self.max_laps)
        d_t = self._compute_offset()

        tile_idx = self._nearest_tile_index()   # tile index drives sector logic
        in_sector, sector_idx = self._sector_state_by_index(tile_idx)
        in_strip = (self._pit_d_min <= d_t <= self._pit_d_max)
        pitroad = bool(in_sector and in_strip)

        # edge logs
        pit_enter = pitroad and not self._prev_pitroad
        pit_exit  = (not pitroad) and self._prev_pitroad
        self._prev_pitroad = pitroad

        # clear lock when we're not in any sector (so next sector can service)
        if not in_sector:
            self._pit_lock_sector = None

        pit_executed = False
        if pit_enter and (velocity < self._pit_speed_max):
            # only service if we haven't serviced this sector yet
            if self._pit_lock_sector is None or self._pit_lock_sector != sector_idx:
                self._pit_lock_sector = sector_idx
                self._fuel = 1.0
                self._wear = 0.0
                pit_executed = True
                print(f"[PIT] SERVICE at sector={sector_idx} ell={ell_t:.3f} d_t={d_t:+.2f}")

        # logs to check edges
        if pit_enter:
            print(f"[PIT] ENTER sector={sector_idx} ell={ell_t:.3f} d_t={d_t:+.2f}")
        if pit_exit:
            print(f"[PIT] EXIT  ell={ell_t:.3f} d_t={d_t:+.2f}")
        # ==================================

        info["pitroad"] = pitroad
        info["pit_enter"] = pit_enter
        info["pit_exit"] = pit_exit
        info["pit_executed"] = pit_executed
        info["ell"] = float(ell_t)
        info["d_t"] = float(d_t)
        info["out_of_fuel"] = bool(out_of_fuel)
        info["tires_gone"] = bool(tires_gone)
        info["fuel"] = float(self._fuel)
        info["wear"] = float(self._wear)

        # Update observation
        return self._get_obs(observation), reward, terminated, truncated, info

    # render()
    def render(self):
        if self.render_mode == "rgb_array":
            return self._env.render()
        out = self._env.render()

        # Update or create the separate gauge window
        try:
            self._ensure_gauge_window()
            self._gauge.update(fuel=self._fuel, tire_health=1.0 - self._wear)
        except Exception:
            pass

        return out

    def close(self):
        try:
            if self._gauge is not None:
                self._gauge.close()
                self._gauge = None
        except Exception:
            pass
        self._env.close()

    # Helper Functions
    def _map_action(self, action):
        # Base CarRacing expects only 3 controls, so we keep 'pit' separate
        if self.continuous:
            a = np.asarray(action, dtype=np.float32)
            if a.shape[0] < 4:
                a = np.concatenate([a, np.array([0.0], dtype=np.float32)], axis=0)
            steer = float(np.clip(a[0], -1.0, 1.0))
            gas   = float(np.clip(a[1],  0.0, 1.0))
            brake = float(np.clip(a[2],  0.0, 1.0))
            pit   = bool(a[3] >= 0.5)

            # ==== wear-dependent steering and throttle ====
            # g(wear) in [steering_grip_min, 1.0]
            grip = 1.0 - self.steering_wear_strength * self._wear
            grip = float(np.clip(grip, self.steering_grip_min, 1.0))
            steer *= grip

            # Throttle (acceleration) also affected by grip
            gas *= grip

            return np.array([steer, gas, brake], dtype=np.float32), pit
        else:
            a = int(action)
            pit = (a == 5)
            if a == 0:   base = np.array([0.0, 0.0, 0.0], dtype=np.float32)
            elif a == 1: base = np.array([+1.0, 0.0, 0.0], dtype=np.float32)
            elif a == 2: base = np.array([-1.0, 0.0, 0.0], dtype=np.float32)
            elif a == 3: base = np.array([0.0, 1.0, 0.0], dtype=np.float32)
            elif a == 4: base = np.array([0.0, 0.0, 1.0], dtype=np.float32)
            elif a == 5: base = np.array([0.0, 0.0, 0.0], dtype=np.float32)
            else: raise ValueError(f"Invalid discrete action {a}")

            # Same grip logic for discrete mode
            grip = 1.0 - self.steering_wear_strength * self._wear
            grip = float(np.clip(grip, self.steering_grip_min, 1.0))
            steer *= grip

            return base, pit

    def _get_obs(self, base_obs):
        d_t = self._compute_offset()
        v_t = self._get_velocity()
        ell_t = self._env.unwrapped.tile_visited_count / len(self._env.unwrapped.track)

        tile_idx = self._nearest_tile_index()
        track_heading = self._track_heading(tile_idx)
        heading_error = self._angle_diff(self._env.unwrapped.car.hull.angle, track_heading)
        _, v_lat = self._velocity_components(track_heading)

        infield = self._is_infield_from_offset()
        pitroad = self._is_in_pit()

        kappa_t = self._compute_lookahead_curvature()

        state_vec = np.array([
            float(np.clip(d_t, -5.0, 5.0)), # d_t: lateral distance from track center
            float(np.clip(v_t, 0.0, 70.0)), # v_t: current velocity
            1.0 if infield else 0.0, # infield: whether the car is in the infield
            1.0 if pitroad else 0.0, # pitroad: whether the car is on the pit road
            float(np.clip(ell_t, 0.0, 1.0)), # ell_t: progress along the track
            float(np.clip(self._wear, 0.0, 1.0)), # w_t: tire wear
            float(np.clip(self._fuel, 0.0, 1.0)), # f_t: fuel level
            float(np.clip(kappa_t, -0.05, 0.05)), # kappa_t: track curvature
            float(np.clip(heading_error, -math.pi, math.pi)), # psi_t: heading error vs track
            float(np.clip(v_lat, -20.0, 20.0)), # v_lat: lateral speed relative to track
        ], dtype=np.float32)

        return {"image": base_obs, "state": state_vec}

    def _compute_offset(self):
        env = self._env.unwrapped

        # Car position in Box2D world coordinates
        car_position = env.car.hull.position
        car_x, car_y = float(car_position[0]), float(car_position[1])

        track = env.track
        if not track or len(track) == 0:
            return 0.0

        # Find the track tile whose center (x, y) is closest to the car
        min_distance_squared = float("inf")
        nearest_tile_index = 0
        for i, (tile_alpha, tile_beta, tile_center_x, tile_center_y) in enumerate(track):
            delta_x = car_x - tile_center_x
            delta_y = car_y - tile_center_y
            distance_squared = delta_x * delta_x + delta_y * delta_y
            if distance_squared < min_distance_squared:
                min_distance_squared = distance_squared
                nearest_tile_index = i

        # Get the center and orientation (beta) of that nearest tile
        _, nearest_tile_beta, nearest_tile_center_x, nearest_tile_center_y = track[nearest_tile_index]

        # Compute vector from centerline point to car
        vector_to_car_x, vector_to_car_y = car_x - nearest_tile_center_x, car_y - nearest_tile_center_y

        # Road normal is (cos(beta), sin(beta)) as used in gym's create_track()
        # It is a vector that points sideways, perpendicular to the direction you are driving
        road_normal_x, road_normal_y = math.cos(nearest_tile_beta), math.sin(nearest_tile_beta)

        # Signed lateral offset = projection of w onto the normal,
        signed_distance = (vector_to_car_x * road_normal_x + vector_to_car_y * road_normal_y)

        return float(np.clip(signed_distance, -5.0, 5.0))

    def _compute_lookahead_curvature(self):
        """
        Estimate signed track curvature ahead of the car by looking at how the
        road orientation (beta) changes over the next few tiles.

        Positive kappa_t  -> turning one way
        Negative kappa_t  -> turning the other way
        """
        env = self._env.unwrapped
        track = getattr(env, "track", None)

        # Need at least 2 tiles to define a turn
        if not track or len(track) < 2:
            return 0.0

        n = len(track)
        lookahead_tiles = 12 # how many segments ahead to average over

        # Start from the tile closest to the car
        idx = self._nearest_tile_index()

        total_delta = 0.0
        samples = 0

        for k in range(lookahead_tiles):
            i0 = (idx + k) % n
            i1 = (idx + k + 1) % n

            _, beta0, _, _ = track[i0]
            _, beta1, _, _ = track[i1]

            d_beta = self._angle_diff(beta1, beta0) # in [-pi, pi]
            total_delta += d_beta
            samples += 1

        if samples == 0:
            return 0.0

        # Average turn per segment (radians)
        avg_delta = total_delta / float(samples)

        # Normalize to [-1, 1] by dividing by pi, then scale down to about [-0.05, 0.05]
        kappa = (avg_delta / math.pi) * 0.05

        return float(kappa)


    def _is_infield_from_offset(self):
        # Placeholder implementation
        # return False
        # use lateral offset d_t from the track centerline. If |d_t| > 4.0 meters, treat as infield/off-track.
        d_t = self._compute_offset()
        return abs(d_t) > 4.0

    def _nearest_tile_index(self):
        """
        Index of the track tile whose center is closest
        to the car (aligns pit logic with painted tiles).
        """
        env = self._env.unwrapped
        track = env.track
        if not track:
            return 0
        car_x, car_y = float(env.car.hull.position[0]), float(env.car.hull.position[1])
        best_i, best_d2 = 0, float("inf")
        for i, (_, _, cx, cy) in enumerate(track):
            dx, dy = car_x - cx, car_y - cy
            d2 = dx*dx + dy*dy
            if d2 < best_d2:
                best_d2, best_i = d2, i
        return best_i

    @staticmethod
    def _angle_diff(a: float, b: float) -> float:
        """
        Smallest signed difference between two angles a and b, in radians,
        wrapped into [-pi, pi].
        """
        d = a - b
        # Wrap using atan2(sin, cos) for numerical stability
        return math.atan2(math.sin(d), math.cos(d))

    def _track_heading(self, tile_idx: int | None = None) -> float:
        """
        Heading of the track centreline at the current tile, derived from the
        direction between consecutive tile centres.
        """
        env = self._env.unwrapped
        track = getattr(env, "track", None)
        if not track or len(track) < 2:
            return 0.0
        if tile_idx is None:
            tile_idx = self._nearest_tile_index()
        n = len(track)
        i_next = (tile_idx + 1) % n
        _, _, cx0, cy0 = track[tile_idx]
        _, _, cx1, cy1 = track[i_next]
        return math.atan2(cy1 - cy0, cx1 - cx0)

    def _velocity_components(self, track_heading: float):
        """
        Resolve car velocity into components aligned with the track heading.
        """
        vx, vy = float(self._env.unwrapped.car.hull.linearVelocity[0]), float(self._env.unwrapped.car.hull.linearVelocity[1])
        v_long = vx * math.cos(track_heading) + vy * math.sin(track_heading)
        v_lat = -vx * math.sin(track_heading) + vy * math.cos(track_heading)
        return v_long, v_lat

    def _sector_state_by_index(self, tile_idx: int):
        """
        Given a tile index, return (in_sector, sector_idx)
        consistent with paint logic where prog = i / n.
        """
        env = self._env.unwrapped
        n = max(1, len(env.track))
        count = max(1, int(self._pit_count))
        step = 1.0 / float(count)                 # sector length in progress units
        width = float(self._pit_sector_width)     # painted width in progress units

        prog = tile_idx / float(n)
        sector_idx = (tile_idx * count) // n
        start = sector_idx * step

        # tiny epsilon eliminates fencepost disagreement
        eps = 1e-9
        in_sector = (start - eps <= prog <= start + width + eps)
        return in_sector, int(sector_idx)

    def _sync_pit_bounds(self):
        """
        Set lateral pit detection [d_min, d_max] to exactly
        match how the pit strip is drawn (centered or sided).
        """
        if self._pit_center_test:
            half = float(abs(self._pit_width)) / 2.0
            self._pit_d_min, self._pit_d_max = -half, +half
        else:
            inner = float(abs(self._pit_inner_offset))
            width = float(abs(self._pit_width))
            d_hi = self._pit_sign * inner
            d_lo = self._pit_sign * (inner + width)
            self._pit_d_min, self._pit_d_max = (min(d_lo, d_hi), max(d_lo, d_hi))

    def _is_in_pit(self, d_t=None, ell_t=None):
        """
        Return True iff current lateral offset is within
        pit bounds and the current tile lies in a pit sector.
        """
        try:
            if d_t is None:
                d_t = self._compute_offset()
            tile_idx = self._nearest_tile_index()
            in_sector, _ = self._sector_state_by_index(tile_idx)
            in_strip = (self._pit_d_min <= d_t <= self._pit_d_max)
            return bool(in_sector and in_strip)
        except Exception:
            return False

    def _get_velocity(self):
        v_t = self._env.unwrapped.car.hull.linearVelocity
        return np.linalg.norm(v_t)

    def _ensure_gauge_window(self):
        if self._gauge is not None:
            return
        try:
            self._gauge = _GaugeWindow(title="Fuel & Tire Gauges", width=280, height=110)
        except Exception:
            self._gauge = None

    def _build_pit_polys(self):
        """
        Build painted pit quads on the track.
        """
        env = self._env.unwrapped
        track = getattr(env, "track", None)
        if not track or len(track) < 2:
            return

        try:
            if self._pit_polys_count > 0:
                env.road_poly = env.road_poly[:-self._pit_polys_count]
                self._pit_polys_count = 0
        except Exception:
            self._pit_polys_count = 0

        n = len(track)
        count = max(1, int(self._pit_count))
        step = 1.0 / float(count)
        width_prog = float(self._pit_sector_width)

        PIT_COLOR = (255, 140, 0) # orange

        def lateral_offset(cx, cy, beta, offset_m):
            nx, ny = math.cos(beta), math.sin(beta)
            return (cx + offset_m * nx, cy + offset_m * ny)

        added = 0
        for i in range(n - 1):
            prog = i / float(n)
            k = int((prog % 1.0) / step)
            start = k * step
            if not (start <= prog <= start + width_prog):
                continue

            _, beta_i, cx_i, cy_i = track[i]
            _, beta_j, cx_j, cy_j = track[i + 1]

            if self._pit_center_test:
                half = float(abs(self._pit_width)) / 2.0
                # left/right edges around the centerline
                ix_i, iy_i = lateral_offset(cx_i, cy_i, beta_i, -half)
                ox_i, oy_i = lateral_offset(cx_i, cy_i, beta_i, +half)
                ix_j, iy_j = lateral_offset(cx_j, cy_j, beta_j, -half)
                ox_j, oy_j = lateral_offset(cx_j, cy_j, beta_j, +half)
            else:
                inner = float(abs(self._pit_inner_offset))
                strip_w = float(abs(self._pit_width))
                # right side using sign
                ix_i, iy_i = lateral_offset(cx_i, cy_i, beta_i, self._pit_sign * inner)
                ox_i, oy_i = lateral_offset(cx_i, cy_i, beta_i, self._pit_sign * (inner + strip_w))
                ix_j, iy_j = lateral_offset(cx_j, cy_j, beta_j, self._pit_sign * inner)
                ox_j, oy_j = lateral_offset(cx_j, cy_j, beta_j, self._pit_sign * (inner + strip_w))

            quad = [(ix_i, iy_i), (ox_i, oy_i), (ox_j, oy_j), (ix_j, iy_j)]
            try:
                env.road_poly.append((quad, PIT_COLOR))
                added += 1
            except Exception:
                break

        self._pit_polys_count = added

class _GaugeWindow:
    def __init__(self, title="Fuel & Tire Gauges", width=280, height=110):
        self._pyglet = pyglet
        self.window = pyglet.window.Window(width=width, height=height, caption=title, resizable=False, vsync=False)
        self.batch = pyglet.graphics.Batch()
        self.shapes = {}
        self.labels = {}

        # Layout
        self.margin = 30
        self.bar_w = width - self.margin*2
        self.bar_h = 16
        self.gap = 30

        # Colors
        self.col_bg   = (25, 25, 25)
        self.col_ok   = (80, 200, 120)
        self.col_warn = (240, 200, 80)
        self.col_crit = (230, 80, 80)
        self.col_text = (240, 240, 240, 255)

        # Draws Fuel bar at the top and Tire Wear bar below
        self._make_row("FUEL", y_top=height - self.margin)
        self._make_row("TIRE WEAR", y_top=height - self.margin - (self.bar_h + self.gap))

        # initial draw
        self.update(1.0, 1.0)

    def _make_row(self, name, y_top):
        x0 = self.margin
        self.shapes[(name, "bg")]   = shapes.Rectangle(x0, y_top - self.bar_h, self.bar_w, self.bar_h, color=self.col_bg, batch=self.batch)
        self.shapes[(name, "fill")] = shapes.Rectangle(x0, y_top - self.bar_h, 1, self.bar_h, color=self.col_ok, batch=self.batch)
        self.labels[(name, "left")] = self._pyglet.text.Label(
            name, font_size=10, color=self.col_text, x=x0, y=y_top + 2,
            anchor_x="left", anchor_y="baseline", batch=self.batch
        )
        self.labels[(name, "right")] = self._pyglet.text.Label(
            "100%", font_size=10, color=self.col_text, x=x0 + self.bar_w, y=y_top - self.bar_h + 2,
            anchor_x="right", anchor_y="baseline", batch=self.batch
        )

    def _colour_format(self, v):
        if v > 0.5: return self.col_ok
        if v > 0.2: return self.col_warn
        return self.col_crit

    def update(self, fuel, tire_health):
        pyglet = self._pyglet

        # Keep window responsive
        pyglet.clock.tick()
        self.window.switch_to()
        self.window.dispatch_events()

        fuel = float(np.clip(fuel, 0.0, 1.0))
        tire = float(np.clip(tire_health, 0.0, 1.0))

        # Update fills + labels
        f_fill = self.shapes[("FUEL", "fill")]
        f_fill.width = int(self.bar_w * fuel)
        f_fill.color = self._colour_format(fuel)

        t_fill = self.shapes[("TIRE WEAR", "fill")]
        t_fill.width = int(self.bar_w * tire)
        t_fill.color = self._colour_format(tire)

        self.labels[("FUEL", "right")].text = f"{int(fuel*100)}%"
        self.labels[("TIRE WEAR", "right")].text = f"{int(tire*100)}%"

        # Draw this frame
        self.window.clear()
        self.batch.draw()
        self.window.flip()

    def close(self):
        try:
            self.window.close()
        except Exception:
            pass


## Env Test

In [ ]:
import os
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

EPISODES = 1
SEED = 0
VIDEO_FOLDER = "videos_custom"
VIDEO_PREFIX = "custom_carracing"

def make_custom_env():
    # IMPORTANT: use render_mode="rgb_array" for Colab + video recording
    env = CarRacing(
        render_mode="rgb_array",
        continuous=True,
        lap_complete_percent=0.95,
        reward_shaping=True,
        max_episode_steps=3000,
        max_laps=3,
    )

    env = RecordVideo(
        env,
        video_folder=VIDEO_FOLDER,
        episode_trigger=lambda ep_id: True,  # record every episode
        name_prefix=VIDEO_PREFIX,
    )
    return env

def run_and_record():
    env = make_custom_env()

    for ep in range(1, EPISODES + 1):
        obs, info = env.reset(seed=SEED)
        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            # random policy for demo
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)

            total_reward += float(reward)
            steps += 1
            done = terminated or truncated

        print(f"Episode {ep} finished: steps={steps}, return={total_reward:.2f}")

    env.close()

    # Show latest recorded video
    if not os.path.exists(VIDEO_FOLDER):
        print("No video folder found:", VIDEO_FOLDER)
        return

    mp4_files = sorted(
        [os.path.join(VIDEO_FOLDER, f) for f in os.listdir(VIDEO_FOLDER) if f.endswith(".mp4")]
    )
    if not mp4_files:
        print("No .mp4 files found in", VIDEO_FOLDER)
        return

    latest_video = mp4_files[-1]
    print("Showing video:", latest_video)
    display(Video(latest_video, embed=True))

run_and_record()


Episode 1 finished: steps=1081, return=-65.40


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Showing video: videos_custom/custom_carracing-episode-0.mp4


# Ziyang's CNN


In [ ]:
# https://medium.com/@samina.amin/deep-q-learning-dqn-71c109586bae
import argparse
from datetime import datetime
import itertools
import random
from collections import deque
import os

import numpy as np
import matplotlib.pyplot as plt

# pip install torch
import torch
import torch.nn as nn
import torch.optim as optim

class RunningMeanStd:
    def __init__(self, shape, eps: float = 1e-4):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = eps

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        batch_mean = np.mean(x, axis=0)
        batch_var = np.var(x, axis=0)
        batch_count = x.shape[0] if x.ndim > 1 else 1

        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + np.square(delta) * self.count * batch_count / tot_count
        new_var = M2 / tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

    def normalize(self, x: np.ndarray, clip_range: float = 5.0):
        normed = (x - self.mean) / (np.sqrt(self.var) + 1e-8)
        if clip_range is not None:
            normed = np.clip(normed, -clip_range, clip_range)
        return normed


class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action_idx, reward, next_state, terminated, truncated):
        self.buffer.append((state, action_idx, reward, next_state, terminated, truncated))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, action_idx, rewards, next_states, terminated, truncated = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(action_idx, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(terminated, dtype=np.float32),
            np.array(truncated, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


def curated_action_set():
    actions = [
        [-0.8, 0.8, 0.0, 0.0],  # hard left, throttle
        [0.0, 0.8, 0.0, 0.0],   # straight, throttle
        [0.8, 0.8, 0.0, 0.0],   # hard right, throttle
        [-0.5, 0.4, 0.0, 0.0],  # medium left, light gas
        [0.0, 0.4, 0.0, 0.0],   # coast with a little gas
        [0.5, 0.4, 0.0, 0.0],   # medium right, light gas
        [-0.8, 0.0, 0.0, 0.0],  # hard left coast
        [0.0, 0.0, 0.0, 0.0],   # full coast
        [0.8, 0.0, 0.0, 0.0],   # hard right coast
        [0.0, 0.5, 0.7, 0.0],   # straight brake
        [-0.6, 0.2, 0.8, 0.0],  # brake left
        [0.6, 0.2, 0.8, 0.0],   # brake right
        [0.0, 0.2, 0.0, 1.0],   # pit entry (slow, pit on)
    ]
    return np.asarray(actions, dtype=np.float32)

def discretize_action_space(action_space, bins_per_dim=(7, 3, 2, 2)):
    low = action_space.low
    high = action_space.high
    action_dim = low.shape[0]

    if len(bins_per_dim) == 1:
        bins_per_dim = bins_per_dim * action_dim
    assert len(bins_per_dim) == action_dim, "bins_per_dim must match action_dim or be length 1"

    grids = []
    for d in range(action_dim):
        grids.append(np.linspace(low[d], high[d], bins_per_dim[d], dtype=np.float32))

    actions = np.array(list(itertools.product(*grids)), dtype=np.float32)
    return actions


class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim: int, num_actions: int):
        super().__init__()
        hidden = 256

        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )

        self.advantage = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_actions),
        )

        self.value = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.feature(x)
        adv = self.advantage(feat)
        val = self.value(feat)
        adv_mean = adv.mean(dim=1, keepdim=True)
        q = val + (adv - adv_mean)
        return q


class DQNCarRacingAgent:
    def __init__(
        self,
        env: CarRacing,
        bins_per_dim=(7, 3, 2, 2),
        gamma: float = 0.99,
        lr: float = 1e-3,
        batch_size: int = 64,
        replay_capacity: int = 10000,
        min_replay_size: int = 1000,
        epsilon_start: float = 1.0,
        epsilon_end: float = 0.05,
        epsilon_decay_steps: int = 50000,
        target_update_freq: int = 1000,
        reward_clip: float = 80.0,
        use_curated_actions: bool = True,
        device: torch.device | None = None,
    ):
        self.env = env
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.replay_capacity = replay_capacity
        self.min_replay_size = min_replay_size
        self.target_update_freq = target_update_freq
        self.reward_clip = reward_clip

        # Device
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = device

        state_dim = env.observation_space["state"].shape[0]

        self.state_low = np.asarray(env.observation_space["state"].low, dtype=np.float32)
        self.state_high = np.asarray(env.observation_space["state"].high, dtype=np.float32)
        self.state_rms = RunningMeanStd(self.state_low.shape)

        if use_curated_actions:
            self.actions_table = curated_action_set()
        else:
            self.actions_table = discretize_action_space(env.action_space, bins_per_dim)
        self.num_actions = self.actions_table.shape[0]

        # Q network + target network
        self.q_net = DuelingQNetwork(state_dim, self.num_actions).to(self.device)
        self.target_q_net = DuelingQNetwork(state_dim, self.num_actions).to(self.device)
        self.target_q_net.load_state_dict(self.q_net.state_dict())
        self.target_q_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=self.lr)
        self.replay = ReplayBuffer(capacity=replay_capacity)

        # Epsilon-greedy
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay_steps = epsilon_decay_steps
        self.global_step = 0
        self.gradient_steps = 0

        # Reward shaping knobs
        self.progress_coef = 60.0
        self.center_penalty = 0.5
        self.offtrack_penalty = 20.0
        self.pit_bonus = 25.0
        self.speed_target = 35.0
        self.speed_coef = 0.05
        self.heading_penalty = 1.2
        self.lateral_penalty = 0.2
        self.lap_bonus = 500.0
        self.alive_bonus = 0.01
        self.prev_ell = 0.0

    # ===== saving ======
    def save_checkpoint(self, checkpoint_path: str, episode: int, episode_returns):
        os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
        ckpt = {
            "episode": episode,
            "global_step": self.global_step,
            "gradient_steps": self.gradient_steps,
            "q_net_state_dict": self.q_net.state_dict(),
            "target_q_net_state_dict": self.target_q_net.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "episode_returns": episode_returns,
            "actions_table": self.actions_table,
        }
        torch.save(ckpt, checkpoint_path)
        print(f"[DQN] Saved checkpoint to {checkpoint_path}")
    @staticmethod
    def _obs_to_state(obs) -> np.ndarray:
        return np.asarray(obs["state"], dtype=np.float32)

    def _obs_to_state(self, obs) -> np.ndarray:
        state = np.asarray(obs["state"], dtype=np.float32)
        state = np.clip(state, self.state_low, self.state_high)
        self.state_rms.update(state)
        normed = self.state_rms.normalize(state, clip_range=5.0)
        return normed

    def _update_epsilon(self):
        self.global_step += 1
        frac = min(1.0, self.global_step / float(self.epsilon_decay_steps))
        self.epsilon = self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

    def select_action(self, state: np.ndarray):
        if random.random() < self.epsilon:
            action_idx = random.randrange(self.num_actions)
        else:
            s_t = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)  # (1, state_dim)
            with torch.no_grad():
                q_values = self.q_net(s_t)  # (1, num_actions)
            action_idx = int(torch.argmax(q_values, dim=1).item())

        return action_idx, self.actions_table[action_idx]

    def _shape_reward(self, obs, next_obs, env_reward: float, info: dict | None):
        if info is None:
            info = {}

        # Base signals from state vector
        d_t = float(next_obs["state"][0])
        v_t = float(next_obs["state"][1])
        off_track = bool(next_obs["state"][2] > 0.5)
        ell = float(next_obs["state"][4])
        heading_err = float(next_obs["state"][8])
        lat_v = float(next_obs["state"][9])

        delta_ell = max(0.0, ell - self.prev_ell)
        progress_r = self.progress_coef * delta_ell

        center_r = -self.center_penalty * abs(d_t)
        speed_r = self.speed_coef * min(1.0, v_t / self.speed_target)
        offtrack_r = -self.offtrack_penalty if off_track else 0.0
        pit_r = self.pit_bonus if info.get("pit_executed", False) else 0.0
        heading_r = -self.heading_penalty * abs(heading_err)
        lateral_r = -self.lateral_penalty * abs(lat_v)
        lap_r = self.lap_bonus if info.get("lap_finished", False) else 0.0

        shaped = (
            env_reward
            + progress_r
            + center_r
            + speed_r
            + offtrack_r
            + pit_r
            + heading_r
            + lateral_r
            + lap_r
            + self.alive_bonus
        )
        shaped = np.clip(shaped, -self.reward_clip, self.reward_clip)
        self.prev_ell = ell
        return float(shaped)

    def train_step(self):
        if len(self.replay) < self.min_replay_size:
            return

        states, action_idx, rewards, next_states, terminated, truncated = self.replay.sample(self.batch_size)

        states_t = torch.tensor(states, dtype=torch.float32, device=self.device)
        action_idx_t = torch.tensor(action_idx, dtype=torch.long, device=self.device)
        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=self.device).unsqueeze(-1)
        next_states_t = torch.tensor(next_states, dtype=torch.float32, device=self.device)
        terminated_t = torch.tensor(terminated, dtype=torch.float32, device=self.device).unsqueeze(-1)
        truncated_t = torch.tensor(truncated, dtype=torch.float32, device=self.device).unsqueeze(-1)
        # Q(s,a) for current network
        q_all = self.q_net(states_t)
        q_sa = q_all.gather(1, action_idx_t.unsqueeze(-1))

        # target Q(s',a')
        with torch.no_grad():
            # Double DQN target: select action with online net, evaluate with target net
            q_next_online = self.q_net(next_states_t)
            next_action = torch.argmax(q_next_online, dim=1, keepdim=True)
            q_next_target = self.target_q_net(next_states_t).gather(1, next_action)

            done_mask = 1.0 - torch.clamp(terminated_t + truncated_t, 0.0, 1.0)
            target = rewards_t + self.gamma * done_mask * q_next_target

        loss = nn.functional.smooth_l1_loss(q_sa, target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        self.gradient_steps += 1
        if self.gradient_steps % self.target_update_freq == 0:
            self.target_q_net.load_state_dict(self.q_net.state_dict())

    def train(self, total_timesteps: int, checkpoint_dir: str | None = None, save_every_episodes: int | None = None):
        obs, _ = self.env.reset()
        state = self._obs_to_state(obs)
        self.prev_ell = float(obs["state"][4])

        episode_reward = 0.0  # shaped return
        episode_env_reward = 0.0
        episode = 0
        episode_returns = []

        for t in range(total_timesteps):
            self._update_epsilon()

            action_idx, action = self.select_action(state)
            next_obs, reward, terminated, truncated, info = self.env.step(action)
            next_state = self._obs_to_state(next_obs)
            done = bool(terminated or truncated)

            shaped_reward = self._shape_reward(obs, next_obs, float(reward), info)
            self.replay.push(state, action_idx, shaped_reward, next_state, float(terminated), float(truncated))
            self.train_step()

            state = next_state
            obs = next_obs
            episode_reward += shaped_reward
            episode_env_reward += float(reward)

            episode_done = bool(terminated or truncated)
            if episode_done:
                episode += 1
                episode_returns.append(episode_env_reward)
                print(f"[DQN] Ep {episode} | step={t+1} | return={episode_env_reward:.2f} | eps={self.epsilon:.3f} | terminated={terminated} truncated={truncated}")
                obs, _ = self.env.reset()
                state = self._obs_to_state(obs)
                self.prev_ell = float(obs["state"][4])
                episode_reward = 0.0
                episode_env_reward = 0.0

                # Periodic checkpoint saving
                if checkpoint_dir is not None and save_every_episodes is not None:
                    if episode % save_every_episodes == 0:
                        ckpt_path = os.path.join(
                            checkpoint_dir,
                            f"dqn_carracing_ep{episode}.pt",
                        )
                        self.save_checkpoint(ckpt_path, episode, episode_returns)



        return episode_returns


def make_env(render_mode=None, seed: int = 0, max_episode_steps: int = 100000) -> CarRacing:
    env = CarRacing(
        render_mode=render_mode,
        continuous=True,
        lap_complete_percent=0.95,
        reward_shaping=True,
        max_episode_steps=max_episode_steps,
    )
    env.reset(seed=seed)
    return env


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--total-timesteps", type=int, default=600000)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument(
        "--bins",
        nargs=4,
        type=int,
        default=[7, 3, 2, 2],
        metavar=("STEER", "GAS", "BRAKE", "PIT"),
        help="Number of discrete bins per action dimension",
    )
    parser.add_argument("--max-episode-steps", type=int, default=100000)
    parser.add_argument("--render", action="store_true")
    parser.add_argument(
        "--grid-actions",
        action="store_true",
        help="Use full discretized action grid instead of curated 13-action set",
    )
    parser.add_argument("--checkpoint-dir", type=str, default="/content/drive/MyDrive/COMP4010-Project/checkpoints_dqn")
    parser.add_argument("--save-every-episodes", type=int, default=50)
    args = parser.parse_args()

    # Generate a unique timestamp for the training run
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_checkpoint_dir = os.path.join(args.checkpoint_dir, f"run_{timestamp}")

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    render_mode = "human" if args.render else None
    # render_mode = "human" # For testing purpose

    env = make_env(
        render_mode=render_mode,
        seed=args.seed,
        max_episode_steps=args.max_episode_steps,
    )


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = DQNCarRacingAgent(
        env,
        bins_per_dim=tuple(args.bins),
        use_curated_actions=not args.grid_actions,
        device=device,
    )

    episode_returns = agent.train(
        total_timesteps=args.total_timesteps,
        checkpoint_dir=run_checkpoint_dir,
        save_every_episodes=args.save_every_episodes,
    )
    env.close()

    # Final checkpoint
    if len(episode_returns) > 0:
        final_ckpt_path = os.path.join(run_checkpoint_dir, "dqn_final.pt")
        agent.save_checkpoint(final_ckpt_path, episode=len(episode_returns), episode_returns=episode_returns)
        print(f"[DQN] Final checkpoint saved to {final_ckpt_path}")

    # Plot and persist learning output for later comparison
    if len(episode_returns) > 0:
        returns_arr = np.asarray(episode_returns, dtype=np.float32)

        plt.figure()
        plt.plot(returns_arr, label="Episode return")

        window = max(1, len(returns_arr) // 20)
        if window > 1:
            kernel = np.ones(window) / float(window)
            smooth = np.convolve(returns_arr, kernel, mode="valid")
            plt.plot(
                np.arange(window - 1, len(returns_arr)),
                smooth,
                label="Moving avg",
            )

        plt.xlabel("Episode")
        plt.ylabel("Return")
        plt.title("DQN on Custom CarRacing")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(run_checkpoint_dir,"dqn_car_racing_returns.png"), dpi=150)
        plt.close()
    else:
        print("No completed episodes -> nothing to plot.")


if __name__ == "__main__":
    import sys
    sys.argv = ['']
    del sys
    main()


[DQN] Ep 1 | step=454 | return=12.89 | eps=0.991 | terminated=True truncated=False
[PIT] ENTER sector=1 ell=0.046 d_t=+3.23
[PIT] EXIT  ell=0.056 d_t=+4.23
[DQN] Ep 2 | step=781 | return=3.03 | eps=0.985 | terminated=True truncated=False
[DQN] Ep 3 | step=1155 | return=9.27 | eps=0.978 | terminated=True truncated=False
[DQN] Ep 4 | step=1659 | return=-8.51 | eps=0.968 | terminated=True truncated=False
[DQN] Ep 5 | step=1878 | return=3.73 | eps=0.964 | terminated=True truncated=False
[PIT] ENTER sector=3 ell=0.130 d_t=+3.41
[PIT] EXIT  ell=0.137 d_t=+3.57
[DQN] Ep 6 | step=2887 | return=15.94 | eps=0.945 | terminated=True truncated=False
[DQN] Ep 7 | step=3312 | return=-1.80 | eps=0.937 | terminated=True truncated=False
[DQN] Ep 8 | step=3756 | return=18.00 | eps=0.929 | terminated=True truncated=False
[DQN] Ep 9 | step=4381 | return=-7.23 | eps=0.917 | terminated=True truncated=False
[PIT] ENTER sector=3 ell=0.130 d_t=+3.27
[PIT] EXIT  ell=0.133 d_t=+5.00
[DQN] Ep 10 | step=5810 | retu

##